# Lab 7 – PySpark SQL i Nowoczesna Hurtownia Danych (Delta Lakehouse – część 1)

W poprzednich zajęciach (Lab 5–6) poznaliśmy architekturę Sparka, API DataFrame oraz wydajny format kolumnowy **Apache Parquet**.  
W praktyce inżynierii danych sam katalog z plikami Parquet w chmurze lub na dysku sieciowym to jednak dopiero **Data Lake** (a bez dyscypliny i kontroli schematu często zamienia się w tzw. **Data Swamp** – bagno danych).

### Czego brakuje „surowemu” Parquetowi, aby pełnił rolę hurtowni danych?
1. **Brak transakcji ACID:** jeśli zadanie Sparka przerwie się w połowie zapisu lub dwa procesy piszą jednocześnie, w katalogu zostają uszkodzone/częściowe pliki.
2. **Brak operacji `UPDATE`, `DELETE`, `MERGE INTO`:** w klasycznym Parquetcie modyfikacja lub usunięcie jednego rekordu wymaga ręcznego wczytania i nadpisania całych partycji.
3. **Brak historii zmian i audytowalności (*Time Travel*):** nadpisanie katalogu (`mode("overwrite")`) bezpowrotnie usuwa poprzedni stan tabeli.
4. **Brak egzekwowania schematu (*Schema Enforcement*):** przypadkowe dopisanie pliku z błędnymi typami kolumn psuje odczyt dla kolejnych procesów.

Rozwiązaniem tych problemów jest architektura **Data Lakehouse** oparta na otwartym formacie **Delta Lake** (alternatywy na rynku to m.in. Apache Iceberg i Apache Hudi). Delta Lake nakłada na standardowe pliki Parquet warstwę transakcyjną (**`_delta_log/`**), łącząc elastyczność i niski koszt jeziora danych z gwarancjami klasycznej hurtowni SQL.

### Plan laboratorium (część 1 z 2)
* **Moduł 0:** Uruchomienie sesji `SparkSession` z rozszerzeniem **Delta Lake 4.4**.
* **Moduł 1:** Od Parquet do Delta Lake – anatomia dziennika transakcyjnego `_delta_log/`.
* **Moduł 2:** **Spark SQL** i katalog hurtowni – widoki tymczasowe vs tabele trwałe (zarządzane i zewnętrzne).
* **Moduł 3:** Operacje DML w praktyce – `DELETE`, `UPDATE`, `MERGE INTO` (idempotentny *Upsert*).
* **Moduł 4:** *Time Travel* (`VERSION AS OF`, `RESTORE`) oraz *Schema Enforcement vs Schema Evolution*.
* **Moduł 5:** **Architektura Medalionowa (Bronze -> Silver)** na wielotabelowym zbiorze *Mini-E-Commerce* (z paczką przyrostową `day2`).
* **Zadania podsumowujące** do samodzielnego wykonania.


## Moduł 0: Uruchomienie sesji `SparkSession` z Delta Lake

Aby Apache Spark 4.1 obsługiwał format `delta` oraz składnię SQL (`USING DELTA`, `MERGE INTO`, `RESTORE TABLE`, `DESCRIBE HISTORY`), konfigurujemy dwa kluczowe parametry katalogu oraz dołączamy bibliotekę `delta-spark` (4.4.1) poprzez `configure_spark_with_delta_pip()`.

> **Wskazówka zasobowa (Codespaces / kontener 2 vCPU, 8 GB RAM):**  
> Ustawiamy `local[2]` oraz `spark.sql.shuffle.partitions=4` (zamiast domyślnych `200` partycji shuffle), co znacząco przyspiesza operacje na małych i średnich zbiorach w środowisku laboratoryjnym.


In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

# Wskazanie bieżącego interpretera Pythona dla Drivera i Executorów Sparka:
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Interpreter Pythona:", sys.executable)
print("Wersja Pythona:", sys.version.split()[0])


In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    DecimalType,
    BooleanType,
    TimestampType,
)
from pyspark.sql.utils import AnalysisException
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

# Wspólny katalog hurtowni na dysku (współdzielony z Lab 8):
LAKEHOUSE_ROOT = Path("../lakehouse_data").resolve()
WAREHOUSE_DIR = LAKEHOUSE_ROOT / "warehouse"
LAKEHOUSE_ROOT.mkdir(parents=True, exist_ok=True)
WAREHOUSE_DIR.mkdir(parents=True, exist_ok=True)

builder = (
    SparkSession.builder
    .master("local[2]")
    .appName("Lab07-Delta-Lakehouse")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.warehouse.dir", WAREHOUSE_DIR.as_uri())
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Wersja Apache Spark:", spark.version)
print("Katalog hurtowni (warehouse):", WAREHOUSE_DIR)
print("Panel diagnostyczny Spark Web UI: http://localhost:4040")


---
## Moduł 1: Od Parquet do Delta Lake – anatomia `_delta_log/`

Zaczniemy od danych pracowników znanych z **Lab 6** (`../lab_06/employee_1m.csv`).  
Jeśli uruchamiasz notatnik w świeżym środowisku Codespaces bez wygenerowanego pliku z Lab 6, poniższy kod automatycznie wygeneruje mniejszą, deterministyczną próbkę (100 tys. wierszy), dzięki czemu notatnik zadziała w 100% samodzielnie.


In [ ]:
# UWAGA - dopasuj ścieżkę do Twojej lokalizacji pliku z przykładowymi danymi, który został utworzony w poprzednim labie
emp_csv_lab6 = Path("../lab_06/employee_1m.csv")

if emp_csv_lab6.exists():
    print(f"Wczytuję zbiór z Lab 6: {emp_csv_lab6}")
    df_emp = (
        spark.read.csv(str(emp_csv_lab6), header=True, inferSchema=True)
        .limit(100_000)
    )
else:
    print("Brak pliku z Lab 6 - generuję deterministyczny zbiór 100 000 pracowników w pamięci...")
    firstnames = F.array(*[F.lit(x) for x in ["Adam", "Katarzyna", "Krzysztof", "Marek", "Aleksandra", "Agata"]])
    lastnames = F.array(*[F.lit(x) for x in ["Kowalski", "Malinowski", "Barański", "Wróblewski", "Pysla", "Nowak"]])
    df_emp = (
        spark.range(1, 100_001)
        .withColumn("firstname", firstnames[(F.col("id") % 6).cast("int")])
        .withColumn("lastname", lastnames[((F.col("id") * 7) % 6).cast("int")])
        .withColumn("age", (18 + (F.col("id") % 47)).cast("int"))
        .withColumn("salary", F.round(F.lit(4200.0) + (F.col("id") % 5000) * 1.35, 2))
    )

df_emp.show(5)
print("Liczba wierszy:", df_emp.count())


### 1.1. Zapis w formacie Delta i inspekcja dziennika transakcyjnego `_delta_log/`

Zapiszmy teraz ten sam zbiór jako tabelę **Delta** w katalogu `../lakehouse_data/demo/employees_delta` i podejrzyjmy, co fizycznie pojawiło się na dysku.


In [ ]:
emp_delta_path = str((LAKEHOUSE_ROOT / "demo" / "employees_delta").resolve())
shutil.rmtree(emp_delta_path, ignore_errors=True)

df_emp.write.format("delta").mode("overwrite").save(emp_delta_path)

print("Zawartość katalogu tabeli Delta:")
for item in sorted(Path(emp_delta_path).iterdir()):
    print(" -", item.name)

log_dir = Path(emp_delta_path) / "_delta_log"
print("\nZawartość _delta_log/:")
for item in sorted(log_dir.iterdir()):
    print(" -", item.name)


In [ ]:
# Podejrzyjmy zawartość pierwszego pliku transakcyjnego 00000000000000000000.json:
commit_0 = log_dir / "00000000000000000000.json"
with open(commit_0, encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        record = json.loads(line)
        key = list(record.keys())[0]
        print(f"Wpis {i} [{key}]:", json.dumps(record, ensure_ascii=False)[:220], "...")


Zwróć uwagę na strukturę commitu `00000000000000000000.json`:
* **`commitInfo`** – metadane transakcji (kto, kiedy, jaką operacją – `WRITE`, liczba wierszy i bajtów),
* **`protocol`** – minimalna wersja czytnika i pisarza protokołu Delta,
* **`metaData`** – schemat tabeli (kolumny i typy w formacie JSON), partycjonowanie oraz konfiguracja,
* **`add`** – dokładna lista plików `.parquet` należących do tej wersji tabeli wraz ze **statystykami `minValues` / `maxValues` / `numRecords`** (dzięki nim Spark pomija nieistotne pliki bez ich otwierania – mechanizm *Data Skipping*!).

### Ćwiczenie utrwalające 1
Dopisz do tabeli w ścieżce `emp_delta_path` **3 nowych pracowników** używając `.write.format("delta").mode("append").save(emp_delta_path)`.  
Następnie w Pythonie wyświetl listę plików w `log_dir` oraz odczytaj zawartość nowego pliku `00000000000000000001.json`. Co zmieniło się względem wersji `0`?


In [ ]:
# Miejsce na rozwiązanie Ćwiczenia 1:


---
## Moduł 2: Spark SQL i Katalog Hurtowni

Spark udostępnia pełny silnik **Spark SQL** zgodny ze standardem ANSI SQL. Warto wyraźnie odróżnić dwa mechanizmy rejestrowania tabel:
1. **Widok tymczasowy (`createOrReplaceTempView`)** – istnieje tylko w pamięci bieżącej sesji `SparkSession`; znika po wywołaniu `spark.stop()`.
2. **Tabela trwała w katalogu (`saveAsTable` / `CREATE TABLE ... USING DELTA`)** – jej definicja jest utrwalona w katalogu hurtowni, a dane leżą w formacie Delta:
   * **Tabela zarządzana (*Managed Table*):** Spark zarządza zarówno metadanymi, jak i plikami w `spark.sql.warehouse.dir`. Polecenie `DROP TABLE` usuwa **również dane z dysku**.
   * **Tabela zewnętrzna (*External Table* z `LOCATION`):** Spark wskazuje jedynie na istniejący katalog z plikami Delta. Polecenie `DROP TABLE` usuwa wpis z katalogu, ale **pliki na dysku pozostają bezpieczne**.


In [ ]:
# 1. Widok tymczasowy w sesji:
df_emp.createOrReplaceTempView("v_employees_tmp")

spark.sql("""
    SELECT lastname,
           COUNT(*) AS n_employees,
           ROUND(AVG(salary), 2) AS avg_salary
    FROM v_employees_tmp
    GROUP BY lastname
    ORDER BY avg_salary DESC
""").show()


In [ ]:
# 2. Utworzenie trwałego schematu (bazy danych) i tabeli zarządzanej Delta w katalogu:
spark.sql("DROP SCHEMA IF EXISTS lab7_demo CASCADE")
shutil.rmtree(WAREHOUSE_DIR / "lab7_demo.db", ignore_errors=True)
spark.sql("CREATE SCHEMA lab7_demo")
spark.sql("USE lab7_demo")

# Zapis DataFrame jako zarządzanej tabeli Delta:
(
    df_emp.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("lab7_demo.employees_delta")
)

print("Tabele w schemacie lab7_demo:")
spark.catalog.listTables("lab7_demo")


In [ ]:
# Inspekcja szczegółów fizycznych tabeli (DESCRIBE DETAIL) oraz historii transakcji (DESCRIBE HISTORY):
spark.sql("DESCRIBE DETAIL lab7_demo.employees_delta").select(
    "format", "name", "location", "numFiles", "sizeInBytes"
).show(truncate=False)

spark.sql("DESCRIBE HISTORY lab7_demo.employees_delta").select(
    "version", "timestamp", "operation", "operationParameters", "operationMetrics"
).show(truncate=False)


### Ćwiczenie utrwalające 2
Utwórz **zewnętrzną tabelę Delta** `lab7_demo.high_earners_ext` wskazującą na katalog `LAKEHOUSE_ROOT / "demo" / "high_earners_ext"` (użyj składni SQL `CREATE TABLE ... USING DELTA LOCATION '...' AS SELECT ...` dla pracowników z `salary > 8000`).  
Sprawdź wynik `DESCRIBE DETAIL`, a następnie wykonaj `DROP TABLE lab7_demo.high_earners_ext` i zweryfikuj w Pythonie, czy pliki na dysku nadal istnieją.


In [ ]:
# Miejsce na rozwiązanie Ćwiczenia 2:


---
## Moduł 3: Operacje DML i transakcyjność ACID (`DELETE`, `UPDATE`, `MERGE INTO`)

W klasycznym Parquetcie usuwanie lub aktualizacja wybranych rekordów oznaczała ręczne przepisanie plików. W Delta Lake operacje **`DELETE`**, **`UPDATE`** oraz **`MERGE INTO`** są atomowymi transakcjami ACID dostępnymi zarówno z poziomu **SQL**, jak i Pythonowego API **`DeltaTable`**.

> **Jak to działa pod spodem?**  
> Pliki Parquet są niemodyfikowalne (*immutable*). Gdy usuwasz lub aktualizujesz wiersze, Delta zapisuje nowe pliki ze zmienionymi wierszami (lub wektory usunięć), a w nowym wpisie w `_delta_log/` atomowo oznacza stare pliki jako `remove` oraz nowe jako `add`. Czytelnicy widzą albo pełny stan sprzed transakcji, albo pełny stan po niej – nigdy stanu pośredniego!


In [ ]:
# 1. DELETE w SQL – usuwamy pracowników poniżej 20 roku życia:
print("Liczba wierszy przed DELETE:", spark.table("lab7_demo.employees_delta").count())
spark.sql("DELETE FROM lab7_demo.employees_delta WHERE age < 20")
print("Liczba wierszy po DELETE:", spark.table("lab7_demo.employees_delta").count())

# 2. UPDATE przez Pythonowe API DeltaTable – 10% podwyżki dla nazwiska 'Pysla':
dt_emp = DeltaTable.forName(spark, "lab7_demo.employees_delta")
dt_emp.update(
    condition=F.col("lastname") == "Pysla",
    set={"salary": F.round(F.col("salary") * 1.10, 2)},
)

spark.sql("""
    SELECT lastname, ROUND(AVG(salary), 2) AS avg_sal
    FROM lab7_demo.employees_delta
    WHERE lastname IN ('Pysla', 'Kowalski')
    GROUP BY lastname
""").show()


In [ ]:
# 3. MERGE INTO (Upsert) – jednoczesna aktualizacja istniejących i dodanie nowych rekordów:
df_changes = spark.createDataFrame(
    [
        (1, "Adam", "Kowalski", 30, 15000.00),       # aktualizacja istniejącego id=1
        (2, "Katarzyna", "Nowak", 29, 14500.00),     # aktualizacja istniejącego id=2
        (999_001, "Jan", "Nowoczesny", 35, 18000.00), # nowy pracownik
    ],
    schema=["id", "firstname", "lastname", "age", "salary"],
)

(
    dt_emp.alias("target")
    .merge(df_changes.alias("src"), "target.id = src.id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

spark.sql("SELECT * FROM lab7_demo.employees_delta WHERE id IN (1, 2, 999001) ORDER BY id").show()


In [ ]:
# Zobaczmy pełną historię operacji i metryki (ile wierszy zmieniono/wstawiono/usunięto):
spark.sql("DESCRIBE HISTORY lab7_demo.employees_delta").select(
    "version", "operation", "operationMetrics"
).show(truncate=False)


### Ćwiczenie utrwalające 3
Przygotuj krótką ramkę `df_hr_batch` zawierającą kolumny `id, firstname, lastname, age, salary, action` (np. 1 rekord z `action='UPDATE'`, 1 rekord z `action='DELETE'` oraz 1 nowy rekord z `id=999002` i `action='INSERT'`).  
Wykonaj pojedynczą operację `MERGE INTO` na `lab7_demo.employees_delta`, która obsłuży wszystkie trzy przypadki (`whenMatchedDelete`, `whenMatchedUpdate`, `whenNotMatchedInsert`).


In [ ]:
# Miejsce na rozwiązanie Ćwiczenia 3:


---
## Moduł 4: *Time Travel* (`VERSION AS OF`, `RESTORE`) oraz Ewolucja Schematu

Ponieważ Delta Lake nie kasuje od razu starych plików Parquet podczas operacji DML, a jedynie zmienia wpisy w `_delta_log/`, możemy:
1. odpytać dowolną wcześniejszą wersję tabeli (**Time Travel**),
2. porównać stan obecny z historycznym (np. w audycie danych),
3. bezpiecznie cofnąć tabelę po błędzie operatora poleceniem **`RESTORE TABLE ... TO VERSION AS OF ...`** (operacja jest również transakcyjna – tworzy nową wersję w historii!).


In [ ]:
# Symulacja błędu: operator uruchamia DELETE zapominając o klauzuli WHERE!
spark.sql("DELETE FROM lab7_demo.employees_delta")
print("Liczba wierszy po przypadkowym DELETE bez WHERE:", spark.table("lab7_demo.employees_delta").count())

# Sprawdźmy ostatnią poprawną wersję w historii:
history_df = spark.sql("DESCRIBE HISTORY lab7_demo.employees_delta").select("version", "operation")
history_df.show(truncate=False)

last_good_version = (
    history_df.filter(F.col("operation") != "DELETE")
    .agg(F.max("version"))
    .collect()[0][0]
)
print("Ostatnia bezpieczna wersja przed błędnym DELETE:", last_good_version)


In [ ]:
# Podgląd danych z wybranej wersji historycznej (Time Travel):
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table("lab7_demo.employees_delta")
print("Liczba wierszy w wersji 0:", df_v0.count())

# Przywrócenie tabeli do wersji sprzed omyłkowego usunięcia:
spark.sql(f"RESTORE TABLE lab7_demo.employees_delta TO VERSION AS OF {last_good_version}")
print("Liczba wierszy po RESTORE:", spark.table("lab7_demo.employees_delta").count())


### 4.1. *Schema Enforcement* vs *Schema Evolution*

Domyślnie Delta Lake **blokuje** zapis ramki o niezgodnym schemacie (*Schema Enforcement*), chroniąc konsumentów hurtowni przed niespodziewanym „rozjechaniem się” kolumn. Jeśli zmiana schematu jest świadoma (np. nowy system źródłowy zaczyna przesyłać dodatkową kolumnę), możemy jawnie włączyć ewolucję schematu opcją **`.option("mergeSchema", "true")`**.


In [ ]:
df_with_bonus = spark.createDataFrame(
    [(999_010, "Ewa", "Zielińska", 31, 11200.00, 1500.00)],
    schema=["id", "firstname", "lastname", "age", "salary", "bonus"],
)

# 1. Próba zapisu bez mergeSchema kończy się kontrolowanym błędem AnalysisException:
try:
    df_with_bonus.write.format("delta").mode("append").saveAsTable("lab7_demo.employees_delta")
except AnalysisException as exc:
    print("Schema Enforcement zadziałał poprawnie! Przechwycony błąd:")
    print(" ->", str(exc).splitlines()[0])

# 2. Świadoma ewolucja schematu z opcją mergeSchema=true:
(
    df_with_bonus.write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("lab7_demo.employees_delta")
)

spark.sql("SELECT * FROM lab7_demo.employees_delta WHERE id IN (1, 999010)").show()


### Ćwiczenie utrwalające 4
Porównaj w jednym zapytaniu lub krótkim skrypcie **średnią pensję (`AVG(salary)`)** oraz **liczbę wierszy** w wersji `0` tabeli `lab7_demo.employees_delta` względem jej aktualnego stanu (użyj składni SQL `VERSION AS OF 0`).


In [ ]:
# Miejsce na rozwiązanie Ćwiczenia 4:


---
## Moduł 5: Architektura Medalionowa (`Bronze` $\rightarrow$ `Silver`) na zbiorze *Mini-E-Commerce*

W ostatnim module zbudujemy realistyczny, wielotabelowy potok **Bronze $
ightarrow$ Silver**, który w **Lab 8** rozwiniemy o model wymiarowy (*Star Schema*, *SCD Type 2*), warstwę **Gold** i analizę w **DuckDB**.

```
[ data/raw/*.csv, *.json ]
           │  (ingest + _ingestion_ts + _source_file)
           ▼
[ Warstwa BRONZE (Delta) ]  ──> Surowa historia bez utraty rekordów
           │  (czyszczenie, typy, deduplikacja, reguły jakości, MERGE day2)
           ▼
[ Warstwa SILVER (Delta) ]  ──> Czyste, spójne tabele gotowe pod hurtownię (Lab 8)
```

Wygenerujmy najpierw dane wejściowe za pomocą skryptu `ecommerce_data_gen.py`.


In [ ]:
from ecommerce_data_gen import generate_ecommerce_dataset

raw_paths = generate_ecommerce_dataset(base_dir="data/raw")


In [ ]:
# Podejrzyjmy surowe dane wejściowe (day1 vs paczka przyrostowa day2):
print("--- orders_day1.csv (pierwsze 3 wiersze) ---")
with open(raw_paths["orders_day1"], encoding="utf-8") as f:
    for _ in range(4):
        print(f.readline().strip())

print("\n--- orders_day2_incremental.csv (zwróć uwagę na nową kolumnę promo_code!) ---")
with open(raw_paths["orders_day2_incremental"], encoding="utf-8") as f:
    for _ in range(4):
        print(f.readline().strip())


### 5.1. Krok Bronze – utrwalenie surowych danych z metadanymi wczytania

Zasada warstwy **Bronze**: nie odrzucamy rekordów i nie zmieniamy ich logiki biznesowej; konwertujemy surowe pliki CSV/JSON do tabel Delta, dodając kolumny audytowe `_ingestion_ts` oraz `_source_file`.


In [ ]:
BRONZE_DIR = LAKEHOUSE_ROOT / "bronze"
SILVER_DIR = LAKEHOUSE_ROOT / "silver"
shutil.rmtree(BRONZE_DIR, ignore_errors=True)
shutil.rmtree(SILVER_DIR, ignore_errors=True)
shutil.rmtree(WAREHOUSE_DIR / "ecommerce_silver.db", ignore_errors=True)

def ingest_to_bronze(df, table_name: str, source_name: str, mode: str = "overwrite", merge_schema: bool = False):
    target = str((BRONZE_DIR / table_name).resolve())
    writer = (
        df.withColumn("_ingestion_ts", F.current_timestamp())
        .withColumn("_source_file", F.lit(source_name))
        .write.format("delta")
        .mode(mode)
    )
    if merge_schema:
        writer = writer.option("mergeSchema", "true")
    writer.save(target)
    return target

df_cust_raw = spark.read.csv(str(raw_paths["customers_day1"]), header=True, inferSchema=True)
df_prod_raw = spark.read.json(str(raw_paths["products_day1"]))
df_ord_raw = spark.read.csv(str(raw_paths["orders_day1"]), header=True, inferSchema=True)

bronze_cust_path = ingest_to_bronze(df_cust_raw, "customers", "customers_day1.csv")
bronze_prod_path = ingest_to_bronze(df_prod_raw, "products", "products_day1.json")
bronze_ord_path = ingest_to_bronze(df_ord_raw, "orders", "orders_day1.csv")

print("Bronze customers:", spark.read.format("delta").load(bronze_cust_path).count())
print("Bronze products :", spark.read.format("delta").load(bronze_prod_path).count())
print("Bronze orders   :", spark.read.format("delta").load(bronze_ord_path).count())


### 5.2. Krok Silver (`day1`) – czyszczenie, reguły jakości, deduplikacja i rejestracja w katalogu

W warstwie **Silver** wykonujemy transformacje porządkujące:
* **`customers`**: usunięcie białych znaków i normalizacja wielkości liter w `city`, zamiana pustych napisów `email` na `NULL`, deduplikacja po `customer_id`, rzutowanie `signup_date` na `DATE`.
* **`products`**: rzutowanie `unit_price` na precyzyjny typ finansowy `DecimalType(10, 2)` oraz kontrola `unit_price > 0`.
* **`orders`**: rzutowanie `order_ts` na `TimestampType`, odfiltrowanie uszkodzonych rekordów (`quantity > 0`, `unit_price > 0`, poprawny `status`), deduplikacja po `order_item_id` oraz wyliczenie wartości pozycji `line_total`.


In [ ]:
spark.sql("DROP SCHEMA IF EXISTS ecommerce_silver CASCADE")
spark.sql("CREATE SCHEMA ecommerce_silver")

# 1. Silver Customers
df_cust_silver = (
    spark.read.format("delta").load(bronze_cust_path)
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("email", F.when(F.trim(F.col("email")) == "", F.lit(None)).otherwise(F.trim(F.col("email"))))
    .withColumn("signup_date", F.to_date("signup_date"))
    .drop("_ingestion_ts", "_source_file")
    .dropDuplicates(["customer_id"])
)
silver_cust_path = str((SILVER_DIR / "customers").resolve())
df_cust_silver.write.format("delta").mode("overwrite").save(silver_cust_path)

# 2. Silver Products
df_prod_silver = (
    spark.read.format("delta").load(bronze_prod_path)
    .withColumn("unit_price", F.col("unit_price").cast(DecimalType(10, 2)))
    .filter(F.col("unit_price") > 0)
    .drop("_ingestion_ts", "_source_file")
    .dropDuplicates(["product_id"])
)
silver_prod_path = str((SILVER_DIR / "products").resolve())
df_prod_silver.write.format("delta").mode("overwrite").save(silver_prod_path)

# 3. Silver Orders (dodajemy kolumnę promo_code=NULL dla spójności lub zostawimy ją dla ewolucji przy day2)
df_ord_silver_d1 = (
    spark.read.format("delta").load(bronze_ord_path)
    .filter((F.col("quantity") > 0) & (F.col("unit_price") > 0) & F.col("status").isNotNull() & (F.trim(F.col("status")) != ""))
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("unit_price", F.col("unit_price").cast(DecimalType(10, 2)))
    .withColumn("discount_pct", F.col("discount_pct").cast(DoubleType()))
    .withColumn("line_total", F.round(F.col("quantity") * F.col("unit_price") * (1.0 - F.col("discount_pct")), 2).cast(DecimalType(12, 2)))
    .drop("_ingestion_ts", "_source_file")
    .dropDuplicates(["order_item_id"])
)
silver_ord_path = str((SILVER_DIR / "orders").resolve())
df_ord_silver_d1.write.format("delta").mode("overwrite").save(silver_ord_path)

# Rejestracja zewnętrznych tabel Delta w schemacie ecommerce_silver:
spark.sql(f"CREATE TABLE ecommerce_silver.customers USING DELTA LOCATION '{Path(silver_cust_path).as_uri()}'")
spark.sql(f"CREATE TABLE ecommerce_silver.products USING DELTA LOCATION '{Path(silver_prod_path).as_uri()}'")
spark.sql(f"CREATE TABLE ecommerce_silver.orders USING DELTA LOCATION '{Path(silver_ord_path).as_uri()}'")

print("Silver customers:", spark.table("ecommerce_silver.customers").count())
print("Silver products :", spark.table("ecommerce_silver.products").count())
print("Silver orders   :", spark.table("ecommerce_silver.orders").count())


### 5.3. Ładowanie przyrostowe (`day2_incremental`) – idempotentny `MERGE INTO` + ewolucja schematu

Teraz nadchodzi paczka danych z kolejnego dnia (`orders_day2_incremental.csv`). Zawiera ona:
* **~15 000 nowych zamówień** z lipca 2026,
* **~5 000 aktualizacji statusów** dla wcześniejszych zamówień `PENDING`/`SHIPPED`,
* **nową kolumnę `promo_code`** (ewolucja schematu!).

Pokażemy, jak wykonać **idempotentny** zapis (wielokrotne uruchomienie dla tej samej paczki daje dokładnie ten sam stan tabeli końcowej).


In [ ]:
df_ord_day2_raw = spark.read.csv(str(raw_paths["orders_day2_incremental"]), header=True, inferSchema=True)

# 1. Dopisanie paczki day2 do warstwy Bronze z ewolucją schematu:
ingest_to_bronze(
    df_ord_day2_raw,
    table_name="orders",
    source_name="orders_day2_incremental.csv",
    mode="append",
    merge_schema=True,
)

# 2. Oczyszczenie paczki day2 według identycznych reguł jak dla day1:
df_ord_day2_clean = (
    df_ord_day2_raw
    .filter((F.col("quantity") > 0) & (F.col("unit_price") > 0) & F.col("status").isNotNull() & (F.trim(F.col("status")) != ""))
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("unit_price", F.col("unit_price").cast(DecimalType(10, 2)))
    .withColumn("discount_pct", F.col("discount_pct").cast(DoubleType()))
    .withColumn("promo_code", F.when(F.trim(F.col("promo_code")) == "", F.lit(None)).otherwise(F.trim(F.col("promo_code"))))
    .withColumn("line_total", F.round(F.col("quantity") * F.col("unit_price") * (1.0 - F.col("discount_pct")), 2).cast(DecimalType(12, 2)))
    .dropDuplicates(["order_item_id"])
)

# 3. Idempotentny MERGE INTO z automatyczną ewolucją schematu (dodanie kolumny promo_code do Silver):
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

def merge_orders_day2():
    # Odświeżamy uchwyt DeltaTable przy każdym uruchomieniu (ważne po ewolucji schematu!):
    dt_silver_orders = DeltaTable.forPath(spark, silver_ord_path)
    (
        dt_silver_orders.alias("t")
        .merge(df_ord_day2_clean.alias("s"), "t.order_item_id = s.order_item_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# Pierwsze uruchomienie:
merge_orders_day2()
print("Liczba zamówień Silver po 1. MERGE:", spark.table("ecommerce_silver.orders").count())

# Drugie uruchomienie (test idempotentności – liczba wierszy nie powinna wzrosnąć!):
merge_orders_day2()
print("Liczba zamówień Silver po 2. MERGE:", spark.table("ecommerce_silver.orders").count())
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false")


In [ ]:
# Sprawdźmy w DESCRIBE HISTORY metryki obu operacji MERGE:
spark.sql("DESCRIBE HISTORY ecommerce_silver.orders").select(
    "version",
    "operation",
    F.col("operationMetrics.numTargetRowsInserted").alias("inserted"),
    F.col("operationMetrics.numTargetRowsUpdated").alias("updated"),
).show(truncate=False)

# Sprawdźmy czy kolumna promo_code jest dostępna w Silver:
spark.sql("SELECT promo_code, COUNT(*) AS cnt FROM ecommerce_silver.orders GROUP BY promo_code ORDER BY cnt DESC").show()


---
## 6. Zadania podsumowujące do samodzielnego wykonania

### Zadanie 1: Audyt i naprawa po błędzie operatora (`RESTORE`)
1. Utwórz roboczą kopię tabeli zamówień Silver w katalogu `LAKEHOUSE_ROOT / "demo" / "orders_sandbox"` (zapisując aktualną zawartość `ecommerce_silver.orders`).
2. Zasymuluj pomyłkę operatora: wykonaj `DELETE` usuwający wszystkie zamówienia ze statusem `'CANCELLED'`.
3. Odczytaj historię zmian tabeli (`DeltaTable.forPath(spark, ...).history()`), zidentyfikuj numer wersji sprzed usunięcia i przywróć tabelę poleceniem `RESTORE` (w SQL lub `dt.restoreToVersion(...)`).
4. Potwierdź, że liczba wierszy ze statusem `'CANCELLED'` wróciła do wartości pierwotnej.

### Zadanie 2: Tabela kwarantanny (*Quarantine / Dead-Letter Table*)
W Kroku 5.2 odfiltrowaliśmy uszkodzone wiersze z `bronze/orders`. W systemach produkcyjnych błędnych danych nie wyrzuca się po cichu – trafiają one do tabeli kwarantanny do analizy przez zespół jakości danych.
1. Wczytaj `bronze/orders` i wyodrębnij wszystkie **niepoprawne** rekordy (`quantity <= 0`, `unit_price <= 0`, pusty/null `status`).
2. Dodaj kolumnę `rejection_reason` opisującą powód odrzucenia (np. za pomocą `F.when(...).when(...)`).
3. Zapisz wynik jako tabelę Delta w `LAKEHOUSE_ROOT / "bronze" / "orders_quarantine"` i wyświetl rozkład przyczyn odrzucenia (`GROUP BY rejection_reason`).

### Zadanie 3: Przyrostowy potok Bronze $
ightarrow$ Silver dla klientów
1. Wczytaj paczkę `data/raw/customers_day2_updates.csv` i dopisz ją do `bronze/customers` z włączoną ewolucją schematu (`mergeSchema=true`, ponieważ pojawia się nowa kolumna `effective_date`).
2. Utwórz roboczą kopię `SILVER_DIR / "customers_task"` na bazie `silver/customers`, a następnie wykonaj na niej idempotentny `MERGE INTO` dla oczyszczonej paczki `day2` (aktualizując `city`, `country`, `segment`, `email` istniejących klientów oraz wstawiając nowych).

### Zadanie 4: Analiza SQL na warstwie Silver
Korzystając z `spark.sql(...)` na tabelach w schemacie `ecommerce_silver` (`orders`, `customers`, `products`):
1. Wyznacz **TOP 5 kombinacji `(country, category)`** pod względem sumarycznego przychodu netto (`SUM(line_total)`) dla zamówień o statusie innym niż `'CANCELLED'`.
2. Policz dla lipca 2026 (`order_ts >= '2026-07-01'`) udział procentowy zamówień złożonych z użyciem kodu promocyjnego (`promo_code IS NOT NULL`) względem wszystkich zamówień z tego miesiąca.


In [ ]:
# Miejsce na rozwiązanie Zadania 1:


In [ ]:
# Miejsce na rozwiązanie Zadania 2:


In [ ]:
# Miejsce na rozwiązanie Zadania 3:


In [ ]:
# Miejsce na rozwiązanie Zadania 4:


In [ ]:
# Po zakończeniu pracy zamykamy sesję Spark:
spark.stop()
print("Sesja Spark została zamknięta.")
